In [2]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
#create a look up table for the original documents based on their id
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [6]:
from client import client
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=client,
)

In [7]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [8]:
#Test it
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it too late to enroll in the course if I just found out about it?',
 'answer_llm': 'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [9]:
assistant.reset_usage()

In [10]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [11]:
#Run rag for all ground truth questions and store the results in a list
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/515 [00:00<?, ?it/s]

In [12]:
# Collect answers from the results into a list
answers = []

for answer_record in results:
    answers.append(answer_record)

In [13]:
# Calculate the total cost of the RAG calls
assistant.total_cost()

0.09772035000000001

In [14]:
# Save the answers to a CSV file
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)